# APEX - Phase 2: Data Cleaning and Processing

**Goal:** turn the raw OpenF1 responses from Phase 1 into reliable, time-aligned
datasets we can build the dashboard on.

This notebook follows the roadmap's Phase 2 task list exactly:

1. Standardize column names and data types.
2. Parse timestamps consistently.
3. Check for duplicated records.
4. Inspect missing values and invalid measurements.
5. Identify valid lap boundaries.
6. Associate telemetry records with the correct lap.
7. Align car telemetry with location data.
8. Connect weather observations to session timestamps.
9. Connect tire stints to lap information.
10. Calculate basic telemetry statistics.
11. Save processed datasets in Parquet format.

All the heavy lifting lives in `src/processing/cleaning.py`, so the same logic can be
reused by the dashboard later. Here we call it step by step and inspect the results.

## Setup

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))

import pandas as pd
import numpy as np

import config
from src.processing import cleaning as cl

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

RAW = config.RAW_DIR
PROC = config.PROCESSED_DIR

raw = {n: pd.read_parquet(RAW / f"{n}.parquet")
       for n in ["meetings","sessions","drivers","laps","car_data","location","weather","stints"]}
print("Loaded raw datasets:")
for n, df in raw.items():
    print(f"  {n:<12} {df.shape}")

Loaded raw datasets:
  meetings     (1, 18)
  sessions     (1, 15)
  drivers      (20, 12)
  laps         (58, 16)
  car_data     (33619, 10)
  location     (34421, 7)
  weather      (148, 10)
  stints       (2, 8)


## 1-2. Standardize types and parse timestamps

OpenF1 returns dates as ISO strings. We parse every date column to timezone-aware **UTC**
datetimes so joins across datasets line up. We also add readable derived columns without
throwing away the raw values.

In [2]:
car = cl.clean_car_data(raw["car_data"])
location = cl.clean_location(raw["location"])
laps = cl.clean_laps(raw["laps"])
weather = cl.clean_weather(raw["weather"])
stints = cl.clean_stints(raw["stints"])

print("car_data date dtype:", car["date"].dtype)
print("laps date_start dtype:", laps["date_start"].dtype)
display(car[["date","speed","throttle","brake","brake_on","rpm","n_gear","drs","drs_active","valid"]].head())

car_data date dtype: datetime64[ns, UTC]
laps date_start dtype: datetime64[ns, UTC]


,date,speed,throttle,brake,brake_on,rpm,n_gear,drs,drs_active,valid
0,2024-12-08 12:06:49.419000+00:00,0,0,0,False,0,0,0,False,True
1,2024-12-08 12:06:49.739000+00:00,0,0,0,False,0,0,0,False,True
2,2024-12-08 12:06:49.899000+00:00,0,0,0,False,0,0,0,False,True
3,2024-12-08 12:06:50.299000+00:00,0,0,0,False,0,0,0,False,True
4,2024-12-08 12:06:50.499000+00:00,0,0,0,False,0,0,0,False,True


## 3. Check for duplicated records

Telemetry can repeat a timestamp. Our cleaners already drop exact timestamp duplicates;
here we confirm how many were removed and that none remain.

In [3]:
print("car_data: raw rows", len(raw['car_data']), "-> clean rows", len(car),
      "| removed", len(raw['car_data']) - len(car))
print("location: raw rows", len(raw['location']), "-> clean rows", len(location),
      "| removed", len(raw['location']) - len(location))
print("duplicate timestamps remaining (car):", int(car['date'].duplicated().sum()))
print("duplicate timestamps remaining (loc):", int(location['date'].duplicated().sum()))

car_data: raw rows 33619 -> clean rows 33583 | removed 36
location: raw rows 34421 -> clean rows 34421 | removed 0
duplicate timestamps remaining (car): 0
duplicate timestamps remaining (loc): 0


## 4. Inspect missing and invalid measurements

We flagged invalid samples in the cleaners (negative/absurd speeds, (0,0) location fixes).
Now we count them. These are kept in the data but marked, so later analysis can exclude
them without losing the raw record.

In [4]:
print("car_data invalid samples:", int((~car['valid']).sum()), "of", len(car))
print("location invalid (0,0 fixes):", int((~location['valid']).sum()), "of", len(location))
print()
print("Missing values per clean dataset:")
for name, df in {"car":car,"location":location,"laps":laps,"weather":weather}.items():
    na = df.isna().sum(); na = na[na>0]
    print(f"  {name:<9}", dict(na) if not na.empty else "none")

car_data invalid samples: 366 of 33583
location invalid (0,0 fixes): 3359 of 34421

Missing values per clean dataset:
  car       none
  location  none
  laps      {'i1_speed': np.int64(9), 'date_end': np.int64(1)}
  weather   none


## 5. Identify valid lap boundaries

Each lap runs from its `date_start` to the next lap's start. We also mark which laps are
**valid for comparison**: they have a real duration and are not pit-out laps.

In [5]:
cols = ["lap_number","date_start","date_end","lap_duration","is_pit_out_lap","is_valid_lap"]
display(laps[cols].head(12))
print("valid laps:", int(laps['is_valid_lap'].sum()), "of", len(laps))
fastest = laps.loc[laps['is_valid_lap'], 'lap_duration'].min()
print("fastest valid lap:", round(float(fastest),3), "s",
      "(lap", int(laps.loc[laps['lap_duration'].eq(fastest),'lap_number'].iloc[0]), ")")

,lap_number,date_start,date_end,lap_duration,is_pit_out_lap,is_valid_lap
0,1,2024-12-08 13:03:35.033000+00:00,2024-12-08 13:05:14.909000+00:00,99.877,False,True
1,2,2024-12-08 13:05:14.909000+00:00,2024-12-08 13:07:09.865000+00:00,114.938,False,True
2,3,2024-12-08 13:07:09.865000+00:00,2024-12-08 13:08:47.958000+00:00,98.051,False,True
3,4,2024-12-08 13:08:47.958000+00:00,2024-12-08 13:10:17.466000+00:00,89.504,False,True
4,5,2024-12-08 13:10:17.466000+00:00,2024-12-08 13:11:47.309000+00:00,89.813,False,True
5,6,2024-12-08 13:11:47.309000+00:00,2024-12-08 13:13:16.685000+00:00,89.412,False,True
6,7,2024-12-08 13:13:16.685000+00:00,2024-12-08 13:14:46.638000+00:00,89.885,False,True
7,8,2024-12-08 13:14:46.638000+00:00,2024-12-08 13:16:16.379000+00:00,89.824,False,True
8,9,2024-12-08 13:16:16.379000+00:00,2024-12-08 13:17:45.821000+00:00,89.445,False,True
9,10,2024-12-08 13:17:45.821000+00:00,2024-12-08 13:19:16.288000+00:00,90.449,False,True


valid laps: 57 of 58
fastest valid lap: 87.765 s (lap 56 )


## 6. Associate telemetry with the correct lap

Using the lap time windows, we tag every car-data sample with the lap it belongs to.
This lets the dashboard show telemetry for a single chosen lap.

In [6]:
car["lap_number"] = cl.assign_lap_to_timestamps(car["date"], laps).values
per_lap = car.dropna(subset=["lap_number"]).groupby("lap_number").size()
print("samples per lap (first 10):")
print(per_lap.head(10))
print("car samples with a lap assigned:", int(car['lap_number'].notna().sum()), "of", len(car))

samples per lap (first 10):
lap_number
1.0     388
2.0     423
3.0     358
4.0     334
5.0     336
6.0     337
7.0     336
8.0     324
9.0     332
10.0    347
dtype: int64
car samples with a lap assigned: 20695 of 33583


## 7. Align car telemetry with location

car_data and location are sampled on slightly different clocks. We match each car sample
to the nearest location sample (within 0.5 s), giving one table with speed AND position.

In [7]:
car_loc = cl.align_car_location(car, location, tolerance_ms=500)
matched = car_loc[["x","y"]].notna().all(axis=1).sum()
print("car samples matched to a location:", int(matched), "of", len(car_loc))
display(car_loc[["date","speed","throttle","brake","rpm","n_gear","x","y","lap_number"]].head())

car samples matched to a location:

 33566 of 33583


,date,speed,throttle,brake,rpm,n_gear,x,y,lap_number
0,2024-12-08 12:06:49.419000+00:00,0,0,0,0,0,0.0,0.0,NaN
1,2024-12-08 12:06:49.739000+00:00,0,0,0,0,0,0.0,0.0,NaN
2,2024-12-08 12:06:49.899000+00:00,0,0,0,0,0,0.0,0.0,NaN
3,2024-12-08 12:06:50.299000+00:00,0,0,0,0,0,0.0,0.0,NaN
4,2024-12-08 12:06:50.499000+00:00,0,0,0,0,0,0.0,0.0,NaN


## 8. Connect weather to session timestamps

Weather is sampled about once a minute. For any moment we take the **nearest** reading.
Here we attach weather to each lap's start time as an example.

In [8]:
lap_weather = cl.attach_weather(laps["date_start"], weather)
laps_w = pd.concat([laps[["lap_number"]].reset_index(drop=True),
                    lap_weather[["air_temperature","track_temperature","humidity",
                                 "wind_speed","is_raining"]].reset_index(drop=True)], axis=1)
display(laps_w.head())

,lap_number,air_temperature,track_temperature,humidity,wind_speed,is_raining
0,1,27.0,32.1,49.0,2.0,False
1,2,27.0,32.2,49.0,1.3,False
2,3,26.9,31.8,49.0,2.0,False
3,4,26.9,31.8,49.0,2.6,False
4,5,26.9,31.7,50.0,0.8,False


## 9. Connect tire stints to laps

Using the stint table we compute estimated tire age for every lap:

`tire_age = lap - stint_start_lap + age_at_stint_start`

In [9]:
rows = []
for lap in laps["lap_number"]:
    info = cl.tire_age_at_lap(int(lap), stints)
    if info:
        rows.append({"lap_number": int(lap), "compound": info["compound"],
                     "stint": info["stint_number"], "tire_age": info["estimated_tire_age"]})
lap_tires = pd.DataFrame(rows)
display(lap_tires.head(12))
print("compounds used:", lap_tires['compound'].unique().tolist())

,lap_number,compound,stint,tire_age
0,1,MEDIUM,1,0
1,2,MEDIUM,1,1
2,3,MEDIUM,1,2
3,4,MEDIUM,1,3
4,5,MEDIUM,1,4
5,6,MEDIUM,1,5
6,7,MEDIUM,1,6
7,8,MEDIUM,1,7
8,9,MEDIUM,1,8
9,10,MEDIUM,1,9


compounds used: ['MEDIUM', 'HARD']


## 10. Calculate basic telemetry statistics

The driver performance numbers from the roadmap, computed on **valid** samples only.

In [10]:
stats = cl.telemetry_stats(car)
for k, v in stats.items():
    print(f"  {k:<18} {v}")

  avg_speed_kmh      131.3


  max_speed_kmh      339
  min_speed_kmh      0
  avg_throttle_pct   41.0
  braking_pct        11.8
  max_rpm            12419
  n_samples          33217


## 11. Save processed datasets

We write the cleaned, aligned tables to `data/processed/` as Parquet. These are what
Phase 3/4 (the dashboard) will load - never the raw API again.

In [11]:
outputs = {
    "car_data_clean": car,
    "location_clean": location,
    "laps_clean": laps,
    "weather_clean": weather,
    "stints_clean": stints,
    "car_location_aligned": car_loc,
    "lap_tire_age": lap_tires,
}
for name, df in outputs.items():
    path = PROC / f"{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"  saved {name:<22} {df.shape} -> {path.name}")

# Save the driver stats as a tiny one-row table too.
pd.DataFrame([stats]).to_parquet(PROC / "driver_stats.parquet", index=False)
print("  saved driver_stats           (1 row)")

  saved car_data_clean         (33583, 14) -> car_data_clean.parquet

  saved location_clean         (34421, 8) -> location_clean.parquet
  saved laps_clean             (58, 18) -> laps_clean.parquet
  saved weather_clean          (148, 11) -> weather_clean.parquet
  saved stints_clean           (2, 8) -> stints_clean.parquet
  saved car_location_aligned   (33583, 17) -> car_location_aligned.parquet
  saved lap_tire_age           (58, 4) -> lap_tire_age.parquet
  saved driver_stats           (1 row)


## Completion check

Phase 2 is done when:

- [x] Timestamps use a consistent format (UTC datetimes everywhere).
- [x] Telemetry records can be associated with laps.
- [x] Missing and invalid values are handled consistently (flagged, not dropped silently).
- [x] Calculated metrics can be reproduced from the processed data.

**Assumptions recorded** (per roadmap rule 7):
- `brake` is on/off (0 or 100); braking % = share of samples with brake > 0.
- `drs_active` = DRS codes {10, 12, 14}; other codes mean off/eligible.
- `n_gear == 0` is neutral/coasting, kept as valid.
- Location (0,0) is a no-fix placeholder, marked invalid.
- Telemetry-to-lap uses backward time windows; car-to-location uses nearest within 0.5 s.
- Weather uses nearest reading in time, never interpolated as exact.